# LifeMate — ZIP output fine-tune để tải về

1. Tạo notebook Kaggle mới, **Add Input → Notebook Output** của hai notebook đã train
   (Save Version có output trước nếu chưa lưu). Chọn CPU; không cần Internet hay cài thư viện.
2. Điền hai đường dẫn ở cell cấu hình bên dưới. Có thể dùng thư mục output được mount
   hoặc thư mục con chứa `best.pt`, `last.pt`, `bpe.model`.
3. **Run All** → mỗi input tạo một ZIP riêng trong `/kaggle/working/finetune_exports`.
   Bấm link cuối notebook hoặc tải ở mục Output/Files của Kaggle.

Giữ **toàn bộ file** trong mỗi thư mục: checkpoint best/last, BPE, cấu hình, split,
metrics, predictions, source và log nếu input có. Notebook chỉ đóng gói, không train,
không chuyển checkpoint thành ONNX hay tự bổ sung file còn thiếu.
`last.pt` cần để resume; dùng đúng `bpe.model` đi kèm model, nhất là nhánh Sherpa mở rộng token số.
ZIP dùng chế độ lưu nhanh, dung lượng gần bằng input; có kiểm tra CRC sau khi ghi.
Giữ notebook/output private như các lần train.


In [ ]:
from pathlib import Path

# Dán đường dẫn từ bảng Input của Kaggle. Đổi label tùy ý (không dấu, không dấu /).
# Nếu chỉ muốn tải một run, xóa dòng còn lại. Có thể thêm nhiều dòng hơn.
INPUT_DIRS = {
    "hynt_finetune": "",       # /kaggle/input/.../lifemate_hynt_finetune_fresh_5epoch_em
    "sherpa_vi_finetune": "",  # /kaggle/input/.../lifemate_sherpa_vi_68m_main_5epoch_em
}
OUTPUT_DIR = Path("/kaggle/working/finetune_exports")


In [ ]:
import json
import re
import shutil
import zipfile
from datetime import datetime, timezone

def make_exports(input_dirs, output_dir):
    output_dir = Path(output_dir).expanduser().resolve()
    if not input_dirs:
        raise ValueError("Điền ít nhất một input directory ở cell cấu hình.")
    plans = []
    for label, raw in input_dirs.items():
        if not re.fullmatch(r"[A-Za-z0-9][A-Za-z0-9_.-]*", label):
            raise ValueError(f"Label không hợp lệ: {label!r}")
        if not str(raw).strip():
            raise ValueError(f"Chưa điền INPUT_DIRS[{label!r}]. Dán đường dẫn từ bảng Input.")
        root = Path(raw).expanduser().resolve(strict=True)
        if not root.is_dir():
            raise NotADirectoryError(root)
        if output_dir == root or root in output_dir.parents:
            raise ValueError("OUTPUT_DIR phải nằm ngoài mọi input để tránh ZIP tự chứa chính nó.")
        files = []
        for path in sorted(root.rglob("*")):
            if path.is_symlink():
                raise ValueError(f"Input có symlink, hãy chọn thư mục file thật: {path}")
            if path.is_file():
                files.append((path, path.relative_to(root).as_posix(), path.stat().st_size))
        if not files:
            raise ValueError(f"Thư mục rỗng: {root}")
        target = output_dir / f"{label}.zip"
        partial = output_dir / f"{label}.zip.partial"
        if target.exists() or partial.exists():
            raise FileExistsError(f"Đã có {target.name} hoặc .partial. Đổi OUTPUT_DIR hoặc label để chạy lại.")
        total = sum(size for _, _, size in files)
        print(f"{label}: {len(files):,} files | {total / 1024**3:.3f} GiB | {root}", flush=True)
        checkpoints = [rel for _, rel, _ in files if Path(rel).name in {"best.pt", "last.pt", "bpe.model"}]
        print("  Checkpoint/tokenizer tìm thấy:", checkpoints or "không thấy; vẫn đóng gói đúng input đã chọn")
        plans.append((label, root, files, total, target, partial))

    output_dir.mkdir(parents=True, exist_ok=True)
    required = sum(plan[3] for plan in plans) * 1.02 + 64 * 1024**2
    free = shutil.disk_usage(output_dir).free
    if free < required:
        raise OSError(f"Không đủ chỗ: trống {free / 1024**3:.2f} GiB, cần khoảng {required / 1024**3:.2f} GiB.")
    archives = []
    for label, root, files, total, target, partial in plans:
        manifest = dict(label=label, source=str(root),
                        created_utc=datetime.now(timezone.utc).isoformat(),
                        total_bytes=total, files=[dict(path=rel, bytes=size) for _, rel, size in files])
        try:
            with zipfile.ZipFile(partial, "w", compression=zipfile.ZIP_STORED,
                                 allowZip64=True, strict_timestamps=False) as archive:
                for index, (path, rel, size) in enumerate(files, 1):
                    if path.stat().st_size != size:
                        raise RuntimeError(f"Input đang thay đổi: {path}. Dùng output đã Save Version.")
                    archive.write(path, arcname=f"{label}/{rel}")
                    if index == 1 or index % 25 == 0 or index == len(files):
                        print(f"{label}: {index}/{len(files)} — {rel}", flush=True)
                archive.writestr("_export_manifest.json", json.dumps(manifest, ensure_ascii=False, indent=2))
            print(f"{label}: kiểm tra CRC toàn bộ ZIP...", flush=True)
            with zipfile.ZipFile(partial) as archive:
                bad = archive.testzip()
                if bad is not None:
                    raise RuntimeError(f"ZIP lỗi CRC: {bad}")
            partial.rename(target)
        except BaseException:
            partial.unlink(missing_ok=True)
            raise
        print(f"OK: {target} ({target.stat().st_size / 1024**3:.3f} GiB)", flush=True)
        archives.append(target)
    return archives

ARCHIVES = make_exports(INPUT_DIRS, OUTPUT_DIR)


In [ ]:
from IPython.display import FileLink, display
import os

print("ZIP đã kiểm tra xong. Bấm để tải, hoặc mở mục Output/Files của Kaggle:")
for path in ARCHIVES:
    display(FileLink(os.path.relpath(path, Path.cwd())))
